# Diagnostic record — MASSIVE load failure (archived 2026-04-24)

Preserved in the repo as a committed record of what went wrong and how it was fixed. Not part of any automatic pipeline.

## 1. Problem this notebook was built to diagnose

During the first Phase 0 Checkpoint 3 eval freeze (2026-04-23), the produced test set was 100% English even though `load_massive()` was supposed to pull 10 locales from the MASSIVE dataset. The per-locale `try/except` inside the loader silently swallowed every error, so the symptom looked like "MASSIVE contributed zero rows" with no visible root cause. This notebook exists to reproduce and isolate the failure without re-running the full freeze.

## 2. What was learned

- **`datasets` 4.0 removed support for dataset loader scripts entirely.** Both `AmazonScience/massive` (official) and `qanastek/MASSIVE` (community mirror) ship a `MASSIVE.py` loader script and are therefore both dead on `datasets==4.0.0`. Attempts 1–4 in this notebook hit the same root cause: `RuntimeError: Dataset scripts are no longer supported, but found MASSIVE.py`.
- **`trust_remote_code=True` is no longer an escape hatch** — in 4.x it's actively rejected with a matching error message.
- **The silent failure in the original loader was a second bug.** `except Exception as e: print(f'  skipped {loc}: {e}')` masked the real error behind sparse log lines that scrolled past other output during the freeze run. The new loader leads with the exception class name and emits a `loaded N/M locales` summary so the pattern can't recur.

## 3. Resolution

- Migrated the freeze-notebook loader to the `mteb/amazon_massive_intent` parquet mirror, which ships no loader script and loads cleanly on `datasets` 4.x. Confirmed working in this notebook's Attempt 5 (all 10 target locales loaded, schema `{'id','label','label_text','text','lang'}` — labels already decoded strings).
- Applied in commit `ec6547f` (`notebooks/phase0_eval_set_freeze.ipynb`): new loader, per-locale 2k sampling, improved error logging.
- See `nico/DECISION_LOG.md` (2026-04-23) for the decision-log entry covering this migration and the related `out_of_scope` intent-label introduction.

# MASSIVE load failure triage

The eval freeze notebook produced 100% English test data, which means the MASSIVE loader silently dropped every non-English locale. Before re-running the full freeze, isolate which failure mode is responsible.

**Run order:** top to bottom, but **stop at the first cell that succeeds** — you don't need to run the subsequent fallback attempts once one of them works. Paste the full output (including any tracebacks) back to chat.

Hypotheses, narrowest first:

1. Attempt 1 — original loader call: `load_dataset('AmazonScience/massive', 'en-US', split='train')`. If this works, the issue is elsewhere (schema, dedup, etc.).
2. Attempt 2 — same plus `trust_remote_code=True`. `datasets` 4.x got stricter about running dataset-authored loader scripts.
3. Attempt 3 — the `'all'` config which returns every locale in one dataset, then filter client-side.
4. Attempt 4 — `qanastek/MASSIVE` mirror as a fallback.

## 1. Config + login

In [1]:
import traceback
import datasets
from datasets import load_dataset

# The 10 locales we want for the multilingual re-freeze
LOCALES = ['en-US','es-ES','fr-FR','de-DE','pt-PT','it-IT','ja-JP','zh-CN','ko-KR','ar-SA']

print(f'datasets library version: {datasets.__version__}')

# Auth — paste your token at the prompt. Uncheck 'Add token as git credential?'.
from huggingface_hub import login
login()

datasets library version: 4.0.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


## 2. Attempt 1 — original per-locale call

Exactly what the eval freeze loader does. If this succeeds, the problem is NOT the load call and we need to look at the silent `skipped {loc}: {e}` catch or at the dedup step.

In [2]:
print('Attempt 1: load_dataset("AmazonScience/massive", "en-US", split="train")')
print('-' * 70)
attempt_1_ok = False
try:
    ds = load_dataset('AmazonScience/massive', 'en-US', split='train')
    attempt_1_ok = True
    print(f'SUCCESS: {len(ds)} rows')
    print(f'Columns: {ds.column_names}')
    print(f'First row: {ds[0]}')
    print()
    # Check the fields the original loader uses
    row = ds[0]
    print('Field access check (matches the original loader):')
    print(f'  row.get("utt"):        {row.get("utt")!r}')
    print(f'  row.get("intent_str"): {row.get("intent_str")!r}')
    print(f'  row.get("intent"):     {row.get("intent")!r}')
    print(f'  row.get("locale"):     {row.get("locale")!r}')
except Exception:
    traceback.print_exc()
    print('\nAttempt 1 failed. Run Attempt 2 (trust_remote_code=True).')

Attempt 1: load_dataset("AmazonScience/massive", "en-US", split="train")
----------------------------------------------------------------------

Attempt 1 failed. Run Attempt 2 (trust_remote_code=True).


Traceback (most recent call last):
  File "/tmp/ipykernel_16111/2216220133.py", line 5, in <cell line: 0>
    ds = load_dataset('AmazonScience/massive', 'en-US', split='train')
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1392, in load_dataset
    builder_instance = load_dataset_builder(
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1132, in load_dataset_builder
    dataset_module = dataset_module_factory(
                     ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1031, in dataset_module_factory
    raise e1 from None
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 989, in dataset_module_factory
    raise RuntimeError(f"Dataset scripts are no longer supported, but found {filename}")
RuntimeError: Dataset scripts are no longer supported, bu

## 3. Attempt 2 — add `trust_remote_code=True`

**Only run if Attempt 1 failed.** `datasets` 4.x requires explicit opt-in to run dataset-authored loader scripts, and MASSIVE ships one.

In [3]:
print('Attempt 2: same as Attempt 1 plus trust_remote_code=True')
print('-' * 70)
attempt_2_ok = False
try:
    ds = load_dataset('AmazonScience/massive', 'en-US', split='train', trust_remote_code=True)
    attempt_2_ok = True
    print(f'SUCCESS: {len(ds)} rows')
    print(f'Columns: {ds.column_names}')
    print(f'First row: {ds[0]}')
except Exception:
    traceback.print_exc()
    print('\nAttempt 2 failed. Run Attempt 3 ("all" config).')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'AmazonScience/massive' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'AmazonScience/massive' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Attempt 2: same as Attempt 1 plus trust_remote_code=True
----------------------------------------------------------------------

Attempt 2 failed. Run Attempt 3 ("all" config).


Traceback (most recent call last):
  File "/tmp/ipykernel_16111/1640673004.py", line 5, in <cell line: 0>
    ds = load_dataset('AmazonScience/massive', 'en-US', split='train', trust_remote_code=True)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1392, in load_dataset
    builder_instance = load_dataset_builder(
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1132, in load_dataset_builder
    dataset_module = dataset_module_factory(
                     ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1031, in dataset_module_factory
    raise e1 from None
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 989, in dataset_module_factory
    raise RuntimeError(f"Dataset scripts are no longer supported, but found {filename}")
RuntimeEr

## 4. Attempt 3 — `'all'` config

**Only run if Attempts 1 and 2 failed.** Loads every locale in one shot, filters client-side. Often more reliable than iterating configs when the per-config registry is broken.

In [4]:
print('Attempt 3: load_dataset("AmazonScience/massive", "all", split="train")')
print('-' * 70)
attempt_3_ok = False
try:
    ds = load_dataset('AmazonScience/massive', 'all', split='train', trust_remote_code=True)
    attempt_3_ok = True
    print(f'SUCCESS: {len(ds)} rows total (all locales)')
    print(f'Columns: {ds.column_names}')
    print(f'First row: {ds[0]}')
    # Count rows per locale we care about
    from collections import Counter
    locale_counts = Counter()
    target = set(LOCALES)
    for row in ds:
        loc = row.get('locale')
        if loc in target:
            locale_counts[loc] += 1
    print('\nRow counts for target locales:')
    for loc in LOCALES:
        print(f'  {loc}: {locale_counts.get(loc, 0)}')
except Exception:
    traceback.print_exc()
    print('\nAttempt 3 failed. Run Attempt 4 (qanastek mirror).')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'AmazonScience/massive' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'AmazonScience/massive' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Attempt 3: load_dataset("AmazonScience/massive", "all", split="train")
----------------------------------------------------------------------

Attempt 3 failed. Run Attempt 4 (qanastek mirror).


Traceback (most recent call last):
  File "/tmp/ipykernel_16111/2626247656.py", line 5, in <cell line: 0>
    ds = load_dataset('AmazonScience/massive', 'all', split='train', trust_remote_code=True)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1392, in load_dataset
    builder_instance = load_dataset_builder(
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1132, in load_dataset_builder
    dataset_module = dataset_module_factory(
                     ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1031, in dataset_module_factory
    raise e1 from None
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 989, in dataset_module_factory
    raise RuntimeError(f"Dataset scripts are no longer supported, but found {filename}")
RuntimeError:

## 5. Attempt 4 — `qanastek/MASSIVE` mirror

**Only run if Attempts 1, 2, and 3 all failed.** Same source data, different packaging.

In [5]:
print('Attempt 4: load_dataset("qanastek/MASSIVE", "en-US", split="train")')
print('-' * 70)
attempt_4_ok = False
try:
    ds = load_dataset('qanastek/MASSIVE', 'en-US', split='train', trust_remote_code=True)
    attempt_4_ok = True
    print(f'SUCCESS: {len(ds)} rows')
    print(f'Columns: {ds.column_names}')
    print(f'First row: {ds[0]}')
except Exception:
    traceback.print_exc()
    print('\nAttempt 4 failed. Paste the full output back to chat to discuss next steps.')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qanastek/MASSIVE' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qanastek/MASSIVE' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Attempt 4: load_dataset("qanastek/MASSIVE", "en-US", split="train")
----------------------------------------------------------------------

Attempt 4 failed. Paste the full output back to chat to discuss next steps.


Traceback (most recent call last):
  File "/tmp/ipykernel_16111/1123585445.py", line 5, in <cell line: 0>
    ds = load_dataset('qanastek/MASSIVE', 'en-US', split='train', trust_remote_code=True)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1392, in load_dataset
    builder_instance = load_dataset_builder(
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1132, in load_dataset_builder
    dataset_module = dataset_module_factory(
                     ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1031, in dataset_module_factory
    raise e1 from None
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 989, in dataset_module_factory
    raise RuntimeError(f"Dataset scripts are no longer supported, but found {filename}")
RuntimeError: Datas

## 6. Attempt 5 — `mteb/amazon_massive_intent` parquet mirror

**Only run if Attempts 1–4 all failed** (which they will, on `datasets==4.0.0` — loader scripts are unsupported).

MTEB's repackaging of MASSIVE ships as standard parquet files with no loader script, so `datasets` 4.x should accept it. Three sub-cells: discover config names, probe schema on English, aggregate row counts across the 10 target languages.

Watch for:
- Config naming: `'en'` vs `'en-US'` vs `'English'` — we try each.
- Label encoding: MTEB sometimes uses integer `label` with a `ClassLabel` feature; we'll confirm and grab the names list if so.
- Field names: `text` / `utt` / `utterance` for the text, `label` / `intent` / `intent_str` for the label.

In [6]:
# Attempt 5a — discover what configs mteb/amazon_massive_intent exposes.
# mteb locale naming convention may differ from MASSIVE's (e.g. 'en' vs 'en-US').

from datasets import get_dataset_config_names

configs = get_dataset_config_names('mteb/amazon_massive_intent')
print(f'{len(configs)} configs available')
print(configs)

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

52 configs available
['default', 'ta', 'is', 'pl', 'zh-CN', 'el', 'ru', 'te', 'cy', 'he', 'de', 'af', 'ml', 'sl', 'vi', 'mn', 'tl', 'it', 'jv', 'sq', 'fa', 'nb', 'km', 'th', 'ja', 'hi', 'id', 'kn', 'fi', 'ur', 'my', 'lv', 'fr', 'ko', 'sw', 'sv', 'nl', 'da', 'ar', 'ms', 'en', 'am', 'pt', 'ka', 'ro', 'tr', 'hu', 'zh-TW', 'bn', 'hy', 'es', 'az']


In [7]:
# Attempt 5b — probe the English config, print schema, inspect label encoding.

probe_candidates = ['en', 'en-US', 'en_US', 'English']
probe_cfg = next((c for c in probe_candidates if c in configs), configs[0] if configs else None)
print(f'Probing config: {probe_cfg!r}')
print('-' * 70)

ds_probe = load_dataset('mteb/amazon_massive_intent', probe_cfg, split='train')
print(f'Rows: {len(ds_probe)}')
print(f'Columns: {ds_probe.column_names}')
print(f'Features: {ds_probe.features}')
print(f'First row: {ds_probe[0]}')
print()

# Check if the label field is integer-encoded with a ClassLabel feature
label_candidates = ['label', 'intent', 'intent_str', 'label_text']
label_field = next((f for f in label_candidates if f in ds_probe.column_names), None)
text_candidates = ['text', 'utt', 'utterance', 'sentence']
text_field = next((f for f in text_candidates if f in ds_probe.column_names), None)

print(f'Detected text field:  {text_field!r}')
print(f'Detected label field: {label_field!r}')

if label_field and hasattr(ds_probe.features[label_field], 'names'):
    names = ds_probe.features[label_field].names
    print(f'Label is integer-encoded ClassLabel with {len(names)} classes.')
    print(f'  First 10 class names: {names[:10]}')
else:
    print('Label is not a ClassLabel (likely already a string).')

Probing config: 'en'
----------------------------------------------------------------------


Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

train/en.json.gz:   0%|          | 0.00/187k [00:00<?, ?B/s]

...

validation/en.json.gz:   0%|          | 0.00/38.3k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Rows: 11514
Columns: ['id', 'label', 'label_text', 'text', 'lang']
Features: {'id': Value('string'), 'label': Value('string'), 'label_text': Value('string'), 'text': Value('string'), 'lang': Value('string')}
First row: {'id': '1', 'label': 'alarm_set', 'label_text': 'alarm_set', 'text': 'wake me up at nine am on friday', 'lang': 'en'}

Detected text field:  'text'
Detected label field: 'label'
Label is not a ClassLabel (likely already a string).


In [8]:
# Attempt 5c — aggregate row counts across the 10 target locales.
# mteb may use 2-letter codes ('en'), ISO regional ('en-US'), or language names ('English')
# as config names. Try each form for each target language.

TARGET_LANGS_2 = ['en', 'es', 'fr', 'de', 'pt', 'it', 'ja', 'zh', 'ko', 'ar']
FALLBACK_FORMS = {
    'en': ['en-US', 'en_US', 'English'],
    'es': ['es-ES', 'es_ES', 'Spanish'],
    'fr': ['fr-FR', 'fr_FR', 'French'],
    'de': ['de-DE', 'de_DE', 'German'],
    'pt': ['pt-PT', 'pt_PT', 'Portuguese'],
    'it': ['it-IT', 'it_IT', 'Italian'],
    'ja': ['ja-JP', 'ja_JP', 'Japanese'],
    'zh': ['zh-CN', 'zh_CN', 'Chinese', 'zh-cn'],
    'ko': ['ko-KR', 'ko_KR', 'Korean'],
    'ar': ['ar-SA', 'ar_SA', 'Arabic'],
}

totals = {}
used_configs = {}
for lang in TARGET_LANGS_2:
    candidates = [lang] + FALLBACK_FORMS.get(lang, [])
    cfg = next((c for c in candidates if c in configs), None)
    if cfg is None:
        print(f'  {lang}: NOT FOUND (tried: {candidates})')
        totals[lang] = None
        continue
    try:
        ds_l = load_dataset('mteb/amazon_massive_intent', cfg, split='train')
        totals[lang] = len(ds_l)
        used_configs[lang] = cfg
        print(f'  {lang} (config={cfg}): {len(ds_l)} rows')
    except Exception as e:
        print(f'  {lang} (config={cfg}): FAILED {type(e).__name__}: {e}')
        totals[lang] = None

ok = sum(1 for v in totals.values() if v is not None)
total_rows = sum(v for v in totals.values() if v is not None)
print(f'\nLoaded {ok}/{len(TARGET_LANGS_2)} target locales')
print(f'Total rows across targets: {total_rows}')
print(f'Config-name mapping used: {used_configs}')

attempt_5_ok = ok == len(TARGET_LANGS_2)

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

  en (config=en): 11514 rows


Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

...

validation/ar.json.gz:   0%|          | 0.00/44.8k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

  ar (config=ar): 11514 rows

Loaded 10/10 target locales
Total rows across targets: 115140
Config-name mapping used: {'en': 'en', 'es': 'es', 'fr': 'fr', 'de': 'de', 'pt': 'pt', 'it': 'it', 'ja': 'ja', 'zh': 'zh-CN', 'ko': 'ko', 'ar': 'ar'}


## 7. Summary

Prints which attempts ran and their result. Paste this block plus the traceback from the first failing attempt back to chat.

In [9]:
print('Summary')
print('=' * 70)
for name, var in [
    ('Attempt 1 (per-locale, default)          ', 'attempt_1_ok'),
    ('Attempt 2 (per-locale, trust_remote_code)', 'attempt_2_ok'),
    ('Attempt 3 ("all" config)                 ', 'attempt_3_ok'),
    ('Attempt 4 (qanastek mirror)              ', 'attempt_4_ok'),
    ('Attempt 5 (mteb parquet mirror)          ', 'attempt_5_ok'),
]:
    if var in dir():
        print(f'  {name} : {"OK" if globals()[var] else "FAIL"}')
    else:
        print(f'  {name} : not run')

Summary
  Attempt 1 (per-locale, default)           : FAIL
  Attempt 2 (per-locale, trust_remote_code) : FAIL
  Attempt 3 ("all" config)                  : FAIL
  Attempt 4 (qanastek mirror)               : FAIL
  Attempt 5 (mteb parquet mirror)           : OK
